# Phase 1 — Defense Matrix (L4, fp16)

Builds on the validated Phase-0 harness. Runs the **banking** suite across a **matrix of black-box defenses** under the strong **`important_instructions`** attack (and optionally more attacks), on **fp16** (L4 has native bf16, fits 7B without quantization), temperature 0.

Outputs one tidy table: **targeted ASR** and **utility-under-attack** per (defense × attack), saved to CSV. Deterministic env-state checks only.

**Carried over from Phase 0 (all still required):**
- `patch_local.py` content-part shim (agentdojo 0.1.30 ↔ modern vLLM), loaded in-subprocess via `-ml`.
- `--model LOCAL` auto-discovers the served model id; base_url hardcoded to `:8000`.
- ASR computed only over records with BOTH `user_task_id` and `injection_task_id` (144 attacked runs).
- `security == True` ⇒ attack succeeded ⇒ that IS the ASR.

**NOT in this notebook yet:** the adaptive attacker (the paper's crux). It is scaffolded at the end as the next milestone — it needs its own focused build, not a blind stub.

Runtime warning: 5 defenses × 1 attack × 144 runs = 720 agent traces. On L4 fp16 budget a few hours. Start with `important_instructions` only, expand once trusted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install (same modern stack as Phase 0) — then RESTART, skip this cell, continue

In [ ]:
# Modern vLLM keeps numpy 2 and parses Qwen2.5; agentdojo installed AFTER so it stays tolerant.
!pip -q install -U vllm
!pip -q install "agentdojo==0.1.30"

# Colab preinstalls a torchaudio built for a different CUDA than vLLM's torch (cu13 vs cu12),
# and torchaudio's __init__ raises a fatal RuntimeError on the mismatch. We never use audio,
# and a matching torchaudio for this torch is nightly-only -> just remove it. Missing torchaudio
# is a caught ImportError downstream, not a crash.
!pip -q uninstall -y torchaudio

import importlib.metadata as md
for p in ("vllm", "torch", "numpy", "transformers", "agentdojo"):
    try: print(p, md.version(p))
    except Exception as e: print(p, "??", e)
print("\n*** RESTART: Runtime -> Restart session, then skip this cell and continue. ***")

## 2. Write the compat shim + fix agentdojo bugs (run this cell)

In [ ]:
patch_src = '''
# agentdojo 0.1.30 sends content parts as {"type":"text","content":...}; modern vLLM's
# OpenAI server requires {"type":"text","text":...} -> every call 400s, ASR reads a fake 0%.
# LocalLLM uses a text protocol (no native tool-calls), so flattening list-content to a
# string is lossless. Optionally override sampling temperature via AGENTDOJO_TEMPERATURE.
import os
import agentdojo.agent_pipeline.llms.local_llm as _L

_orig_ccr = _L.chat_completion_request
def _ccr(client, model, messages, **kw):
    fixed = []
    for m in messages:
        c = m.get("content")
        if isinstance(c, list):
            c = "".join((p.get("text", p.get("content", "")) if isinstance(p, dict) else str(p)) for p in c)
            m = {**m, "content": c}
        fixed.append(m)
    return _orig_ccr(client, model=model, messages=fixed, **kw)
_L.chat_completion_request = _ccr

_temp = os.environ.get("AGENTDOJO_TEMPERATURE")
if _temp is not None:
    _orig_init = _L.LocalLLM.__init__
    def _init(self, client, model, temperature=0.0, top_p=0.9):
        _orig_init(self, client, model, temperature=float(os.environ["AGENTDOJO_TEMPERATURE"]), top_p=top_p)
    _L.LocalLLM.__init__ = _init
    print(f"[patch_local] temperature override -> {_temp}")

print("[patch_local] content-parts -> string applied inside benchmark process")
'''
with open("patch_local.py", "w") as f:
    f.write(patch_src)
print("wrote patch_local.py")

# --- Fix agentdojo 0.1.30 spotlighting recursion bug (edit installed source once) ---
# Builder does: tool_output_formatter = lambda result: f"<<{tool_output_formatter(result)}>>"
# The lambda references the name it's bound to -> infinite recursion -> RecursionError.
# Capture the original via a default arg. Persists to disk so every subprocess sees it.
# Idempotent: safe to re-run; re-applies after any agentdojo reinstall.
import agentdojo.agent_pipeline.agent_pipeline as _ap, pathlib
_p = pathlib.Path(_ap.__file__)
_src = _p.read_text()
_bug = 'tool_output_formatter = lambda result: f"<<{tool_output_formatter(result)}>>"'
_fix = '__of = tool_output_formatter; tool_output_formatter = lambda result, __of=__of: f"<<{__of(result)}>>"'
if _bug in _src:
    _p.write_text(_src.replace(_bug, _fix)); print("patched spotlighting recursion bug")
elif "__of=__of" in _src:
    print("spotlighting recursion bug already patched")
else:
    print("WARN: spotlighting bug line not found — inspect agent_pipeline.py near the '<<...>>' formatter")

## 3. Confirm this version's defenses / attacks / suites

In [ ]:
from agentdojo.attacks.attack_registry import ATTACKS
print("ATTACKS:", list(ATTACKS.keys()))
# Defenses are a fixed --defense choice set in this version:
#   tool_filter | transformers_pi_detector | spotlighting_with_delimiting | repeat_user_prompt
# plus NO --defense (undefended baseline). Confirm against `--help` if unsure:
!python -m agentdojo.scripts.benchmark --help 2>&1 | grep -A2 -- '--defense'

## 4. Serve Qwen2.5-7B-Instruct in fp16 (L4, bf16)

No quantization — L4's 24GB fits the 7B in bf16 with room for KV cache. If you add the `transformers_pi_detector` defense (a HF classifier ~0.5GB), it still fits. Drop `--gpu-memory-utilization` to 0.80 if the classifier + vLLM contend.

In [ ]:
import subprocess, time, urllib.request, json, os

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # fp16, ungated. Swap for Llama/Mistral/etc. later.
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--port", str(PORT),
    "--dtype", "bfloat16",             # L4 supports bf16 natively
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.85",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",   # Qwen2.5; use 'llama3_json' for Llama-3.x
]
logf = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

def wait_ready(url, proc, timeout=1800):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            raise RuntimeError("vLLM died:\n" + open("vllm.log").read()[-3000:])
        try:
            with urllib.request.urlopen(url + "/models", timeout=5) as r:
                if r.status == 200:
                    print("READY:", json.loads(r.read())["data"][0]["id"]); return
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError("vLLM not ready; see vllm.log")
wait_ready(BASE_URL, server)

## 5. Matrix config

In [ ]:
SUITE    = "banking"
ATTACKS  = ["important_instructions"]            # add "ignore_previous", "direct" to widen
# tool_filter is EXCLUDED: agentdojo hard-couples it to OpenAI function-calling
# ('Tool filter is only supported for OpenAI models'); it cannot run on a local model.
# Run it later only on an OpenAI comparison point, if added.
DEFENSES = [None,                                # undefended baseline (no --defense flag)
            "spotlighting_with_delimiting",
            "repeat_user_prompt",
            "transformers_pi_detector"]

# Variance: at temperature 0 the model is deterministic, so seeds give IDENTICAL results.
# For variance bands, set TEMPERATURE > 0 and SEEDS = [0,1,2] (each seed = one full repeat).
# Default: the reproducible temp-0 point estimate. Report bootstrap CI over tasks for error bars.
TEMPERATURE = 0.0
SEEDS = [0]

print(f"{len(DEFENSES)} defenses x {len(ATTACKS)} attacks x {len(SEEDS)} seed(s) = "
      f"{len(DEFENSES)*len(ATTACKS)*len(SEEDS)} configs x 144 runs")

## 6. Run the matrix

Each (defense × attack × seed) writes to its own logdir so results never collide. `-ml patch_local` loads the shim in-process; `PYTHONPATH` makes it importable. Long-running — a per-config heartbeat is printed.

In [ ]:
import subprocess, os, time

os.makedirs("runs_matrix", exist_ok=True)
env = {**os.environ, "PYTHONPATH": os.getcwd()}

def run_one(defense, attack, seed):
    tag = f"def-{defense or 'none'}_atk-{attack}_seed-{seed}"
    logdir = os.path.join("runs_matrix", tag)
    cmd = ["python", "-m", "agentdojo.scripts.benchmark",
           "--model", "LOCAL", "-s", SUITE, "--attack", attack,
           "--logdir", logdir, "-ml", "patch_local", "-f"]
    if defense is not None:
        cmd += ["--defense", defense]
    e = dict(env)
    if TEMPERATURE and TEMPERATURE > 0:
        e["AGENTDOJO_TEMPERATURE"] = str(TEMPERATURE)
    t0 = time.time()
    # capture output; print only the final summary lines to keep the log readable
    p = subprocess.run(cmd, env=e, capture_output=True, text=True)
    tail = "\n".join(p.stdout.strip().splitlines()[-4:])
    print(f"[{tag}]  {time.time()-t0:6.0f}s\n{tail}\n{'-'*60}")
    if p.returncode != 0:
        print("  STDERR tail:", "\n".join(p.stderr.strip().splitlines()[-8:]))
    return logdir

for attack in ATTACKS:
    for defense in DEFENSES:
        for seed in SEEDS:
            run_one(defense, attack, seed)
print("MATRIX DONE.")

## 7. Aggregate → tidy table (ASR + utility per defense × attack)

Same deterministic rule as Phase 0: count only records with BOTH ids (144 attacked runs); `security==True` ⇒ attack succeeded ⇒ ASR.

In [ ]:
import json, glob, os
import pandas as pd

def as_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return bool(v)
    if isinstance(v, str): return v.strip().lower() in ("true", "1", "yes")
    return False

rows = []
for tag_dir in sorted(glob.glob("runs_matrix/*")):
    tag = os.path.basename(tag_dir)
    recs = []
    for path in glob.glob(os.path.join(tag_dir, "**", "*.json"), recursive=True):
        try:
            with open(path) as f: recs.append(json.load(f))
        except Exception: pass
    attacked = [r for r in recs if r.get("user_task_id") and r.get("injection_task_id")]
    if not attacked:
        print("WARN empty:", tag); continue
    n = len(attacked)
    # tag = def-<d>_atk-<a>_seed-<s>
    d = tag.split("_atk-")[0].replace("def-", "")
    a = tag.split("_atk-")[1].split("_seed-")[0]
    s = tag.split("_seed-")[1]
    rows.append({
        "defense": d, "attack": a, "seed": s, "n": n,
        "ASR": sum(as_bool(r.get("security")) for r in attacked) / n,
        "utility": sum(as_bool(r.get("utility")) for r in attacked) / n,
    })

df = pd.DataFrame(rows).sort_values(["attack", "ASR"]).reset_index(drop=True)
df.to_csv("phase1_results.csv", index=False)
print("saved phase1_results.csv")
# mean over seeds if >1
summary = (df.groupby(["attack", "defense"])[["ASR", "utility"]]
             .mean().round(4).sort_values(["attack", "ASR"]))
print(summary)

## 8. Read the table

- **ASR column** = static targeted attack-success under each defense. Lower = defense reduced the static attack.
- **utility column** = did the real task still complete under attack. A defense that tanks utility is over-defending.
- The `none` row is your undefended baseline; every defense is judged relative to it.

**This is the static result. Per the project's one non-negotiable principle, a static-only table is treated as already-refuted.** The point of Phase 1 is to feed these into the adaptive attacker next: a defense that drops static ASR to ~0 but collapses under an adaptive attacker is exactly the finding the paper exists to quantify.

## 9. NEXT MILESTONE — Adaptive attacker (the crux, its own build)

Not implemented here on purpose — a blind stub would be worse than none. The plan, so the next session starts clean:

**Approach (AutoDojo-style, black-box):** register a custom attack via agentdojo's `-ml` hook (subclass the attack base, same mechanism as the built-ins). For each (defense, task): an **attacker LLM** (local Qwen as a second served model, or the same endpoint) proposes an injection string; run it through the benchmark; read the deterministic `security` outcome; feed success/failure back to the attacker to iterate for K rounds; keep the best. Report adaptive ASR vs the static ASR from §7.

**Why it's separate:** it needs an optimization loop, a per-defense query budget (K), and careful logging of the attack transcript — plus a decision on the attacker model. Build and validate it on ONE defense first — **`transformers_pi_detector`**, the only defense that reduced static ASR here (11.1% → 5.6%), so it's the most interesting to try to break — exactly like Phase 0 validated the harness on one config.

**Report break-out (from CLAUDE.md):** split results by task-specification precision (fully-specified vs action-open) — action-open tasks are far more vulnerable adaptively.

Also queued: InjecAgent cross-check, the full model matrix (Llama-3.1-8B once license cleared, Mistral-7B, Gemma-3-4B, Meta-SecAlign-8B), and temp>0 × 3-seed variance bands.

**Static results so far (banking · important_instructions · Qwen2.5-7B fp16):**
| defense | ASR | utility |
|---|---|---|
| transformers_pi_detector | 5.56% | 31.25% |
| none (baseline) | 11.11% | 34.03% |
| repeat_user_prompt | 12.50% | 32.64% |
| spotlighting_with_delimiting | *(pending clean run)* | |

`tool_filter` excluded (OpenAI-only). Note: absolute utility is capped ~31–34% by agentdojo's LocalLLM text tool-protocol (parse misses), independent of precision or defense — the relative defense comparison is what's meaningful.

In [ ]:
# Free the GPU when done.
try:
    server.terminate(); server.wait(timeout=30); print("vLLM stopped.")
except Exception as e:
    print("killing:", e); server.kill()